# 🏋️‍♂️ GymSync - Google Colab GPU Ollama Runner (Qwen 2.5)
Run Ollama with high-speed GPU acceleration on Google Colab and tunnel it directly to your GymSync backend.

### Step 1: Enable GPU
1. Click **Runtime** on the top menu -> **Change runtime type**
2. Hardware accelerator: Choose **T4 GPU**
3. Click **Save**

In [ ]:
# Step 2: Install Ollama and Cloudflared Tunnel
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("✅ Ollama and Cloudflared installed successfully!")

In [ ]:
# Step 3: Start Ollama with open CORS & Cloudflare Tunnel
import os
import subprocess
import time
import re

print("1. Terminating any old Ollama processes...")
subprocess.run(["pkill", "-f", "ollama"], check=False)
time.sleep(2)

print("2. Starting Ollama daemon with open CORS & 0.0.0.0 binding...")
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_ORIGINS"] = "*"

ollama_proc = subprocess.Popen(["ollama", "serve"], env=env)
time.sleep(4)

print("3. Downloading Qwen 2.5 (7B) to GPU VRAM...")
subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=True)

print("4. Starting Cloudflare Tunnel with host header rewrite...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434", "--http-host-header", "127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline()
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print("\n" + "="*70)
    print("🎉 SUCCESS! YOUR OLLAMA GPU INSTANCE IS LIVE AND READY!")
    print(f"🔗 Public URL: {public_url}")
    print("="*70)
    print("\n📋 Copy this into your GymSync backend/.env file:")
    print("--------------------------------------------------")
    print("ENABLE_OLLAMA=true")
    print(f"OLLAMA_HOST={public_url}")
    print("OLLAMA_MODEL=qwen2.5:7b")
    print("OLLAMA_NUM_CTX=2048")
    print("--------------------------------------------------")
    print("\n💡 Test connection on your local machine by visiting:")
    print(f"http://localhost:5000/api/ai/ollama-status")
    print("="*70)
else:
    print("⚠️ Could not automatically detect tunnel URL. Check logs above.")

# Keep cell active
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Tunnel stopped.")